# RAG CoT
(Chain of Thought)

RAG 파이프라인에서 LLM이 단순한 정보 조합을 넘어서 단계적 사고를 통해 논리적 답변을 할 수 있도록 한다.

In [3]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://eu.api.smith.langchain.com'
os.environ['LANGSMITH_API_KEY'] = os.getenv('langsmith_key')
os.environ['LANGSMITH_PROJECT'] = 'skn23-langchain'
os.environ['OPENAI_API_KEY'] = os.getenv("openai_key")

In [4]:
from langchain_core.documents import Document

def retirver_vertorDB(query=None):
    return [
        Document(page_content="대한민국의 수도는 서울입니다. 서울은 한강을 끼고 발달한 도시입니다."),  # 서울 기본 정보
        Document(page_content="서울의 대표적 관광지는 경복궁, 남산타워, 명동 등이 있습니다."),  # 서울 관광 정보
        Document(page_content="서울의 인구는 약 천만 명이며, 교통·문화 인프라가 잘 갖춰져 있습니다.")  # 서울 인구/인프라 정보
    ]
    
retirver_vertorDB()

[Document(metadata={}, page_content='대한민국의 수도는 서울입니다. 서울은 한강을 끼고 발달한 도시입니다.'),
 Document(metadata={}, page_content='서울의 대표적 관광지는 경복궁, 남산타워, 명동 등이 있습니다.'),
 Document(metadata={}, page_content='서울의 인구는 약 천만 명이며, 교통·문화 인프라가 잘 갖춰져 있습니다.')]

In [14]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.chat_models import init_chat_model

prompt = PromptTemplate.from_template('''  # 검색 문서 기반 답변 지시 프롬프트
당신은 데이터를 분석해서 논리적인 결론을 도출하는 전문가 챗봇입니다.
아래 [검색된 문서]를 바탕으로 사용자의 [질문]에 대해 답변하세요.

[검색된 문서]
{context}

[질문]
{question}

[지시사항]
다음의 단계에 따라 사고한 후, 답변을 작성하세요.
1. **핵심데이터 정리**: 문서에서 사용자 질문과 관련한 팩트를 추출해보세요.
2. **상호관계 분석**: 각 항목별로 어떤 상관관계/시너지를 도출하는지 고민하세요.
3. **논리적 서술**: 위의 사고한 내용을 토대로 사용자 질문에 대한 답변을 준비하세요.
4. **최종 답변**: 서론-본론-결론 구조에 맞춰 완성된 답변을 작성하세요.
''')

llm=init_chat_model('gpt-4.1-mini')
output_parser=StrOutputParser()

chain = prompt | llm | output_parser

question = "서울의 인구, 관광지, 교육인프라를 종합해서 여행하기 좋은 이율르 논리적으로 설명해주세요."

retirver_docs=retirver_vertorDB(question) # 더미 벡터 검색 수행
context= '\n\n'.join([doc.page_content for doc in retirver_docs])  # retrieved_docs에서 page content만 뺴서 하나의 텍스트로 병합

response= chain.invoke({'context':context, 'question':question})      # 컨텍스트 + 질문으로 답변 생성
print(response)

1. **핵심데이터 정리**  
- 서울 인구: 약 천만 명  
- 관광지: 경복궁, 남산타워, 명동 등 대표 관광명소 다수  
- 교육인프라: 문서에 직접적 언급은 없으나, "교통·문화 인프라가 잘 갖춰져 있다"는 정보로 유추 가능  

2. **상호관계 분석**  
- 서울의 대규모 인구는 도시의 다양한 서비스와 인프라를 뒷받침하며, 관광객에게도 편리한 접근성과 다양한 편의를 제공한다.  
- 대표 관광지들은 서울의 풍부한 역사·문화적 자산을 보여주며, 잘 발달한 교통·문화 인프라는 관광객이 여러 장소를 쉽게 이동하며 체험할 수 있게 한다.  
- 교육 인프라는 직접 언급되지 않았으나, 서울이 대한민국의 수도이자 인구 밀집 지역인 점을 고려하면, 우수한 교육 시설 및 정보 인프라가 구축되어 있을 가능성이 높다. 이는 여행객들이 문화적·교육적으로도 높은 수준의 환경을 접할 수 있다는 점에서 긍정적으로 작용한다.  

3. **논리적 서술**  
서울은 약 천만 명의 대규모 인구가 거주하는 도시로, 이는 서울이 충분한 인프라와 서비스를 갖추고 있음을 의미한다. 이와 더불어 경복궁, 남산타워, 명동 등 다양한 대표 관광지가 위치하여 관광객들의 흥미를 돋운다. 교통과 문화 인프라가 잘 구축되어 있어 관광객들이 편리하게 여러 명소를 방문할 수 있다. 또한, 서울은 교육 인프라도 우수하여 문화 체험과 더불어 교육적 가치도 높다. 이들 요소가 상호 보완적으로 작용하여 서울은 종합적으로 여행하기 좋은 도시임을 입증한다.  

4. **최종 답변**  

서울은 약 천만 명의 인구가 거주하는 대한민국의 수도로서, 이는 잘 발달된 도시 인프라와 다양한 편의시설을 갖춘 배경이 됩니다. 대표 관광지인 경복궁, 남산타워, 명동은 서울의 역사적, 문화적 매력을 대변하며, 교통 및 문화 인프라가 잘 갖추어져 있어 관광객들이 쉽고 편리하게 이들 명소를 경험할 수 있습니다. 비록 교육 인프라에 대한 직접적 언급은 없지만, 서울이 국가의 중심지로서 풍부한 문화와 높은 교육 수준을 보유하고